In [11]:
import pandas as pd
import numpy as np

from scipy.stats import wasserstein_distance

In [12]:
REFERENCE_CSV = (
    "../../data/processed/CTB/"
    "s5_sample_20.000_eventlog_one_block_binned_target_features.csv"
)

SIM_CSV = (
    "../../data/processed/CTB/prosit_simulations/sim_log_s5_baseline_sample_20.000_v3_binning.csv"
)

ref = pd.read_csv(REFERENCE_CSV)
sim = pd.read_csv(SIM_CSV)

print(
    f"Reference events: {len(ref):,}"
)

print(
    f"Simulation events: {len(sim):,}"
)

Reference events: 61,718
Simulation events: 48,290


In [13]:
# ==========================================================
# CALCULATE SIMULATION KPIS
# ==========================================================

for col in [
    "enabled:timestamp",
    "start:timestamp",
    "time:timestamp"
]:
    sim[col] = pd.to_datetime(sim[col])

# ----------------------------------------------------------
# EVENT LEVEL KPIs
# ----------------------------------------------------------

sim["waiting_time"] = (
    sim["start:timestamp"]
    - sim["enabled:timestamp"]
).dt.total_seconds() / 60

sim["service_time"] = (
    sim["time:timestamp"]
    - sim["start:timestamp"]
).dt.total_seconds() / 60

# ----------------------------------------------------------
# CASE LEVEL KPI
# ----------------------------------------------------------

case_start = (
    sim.groupby(
        "case:concept:name"
    )["start:timestamp"]
    .transform("min")
)

case_end = (
    sim.groupby(
        "case:concept:name"
    )["time:timestamp"]
    .transform("max")
)

sim["turnaround_time"] = (
    case_end
    - case_start
).dt.total_seconds() / 60

# ----------------------------------------------------------
# RMG EVENTS ONLY
# ----------------------------------------------------------

rmg_activities = [
    "RMG_receive",
    "RMG_delivery",
    "RMG_mixed"
]

sim_rmg = sim[
    sim["concept:name"]
    .isin(rmg_activities)
].copy()

ref_rmg = ref[
    ref["concept:name"]
    .isin(rmg_activities)
].copy()

print(
    f"Reference RMG events: {len(ref_rmg):,}"
)

print(
    f"Simulation RMG events: {len(sim_rmg):,}"
)

Reference RMG events: 20,000
Simulation RMG events: 16,071


In [14]:
#MAP RECEIVE; DELIVER & MIXED
def add_operation_type(df):

    df = df.copy()

    df["operation_type"] = np.where(
        df["concept:name"].str.contains(
            "receive",
            case=False,
            na=False
        ),
        "receive",
        np.where(
            df["concept:name"].str.contains(
                "delivery",
                case=False,
                na=False
            ),
            "delivery",
            np.where(
                df["concept:name"].str.contains(
                    "mixed",
                    case=False,
                    na=False
                ),
                "mixed",
                np.nan
            )
        )
    )

    return df

ref = add_operation_type(ref)
sim = add_operation_type(sim)

In [15]:
#COMPARING FUNCTION'
def evaluate_kpi(
    ref_values,
    sim_values,
    kpi_name,
    segment
):

    ref_values = (
        pd.Series(ref_values)
        .dropna()
    )

    sim_values = (
        pd.Series(sim_values)
        .dropna()
    )

    return {
        "segment": segment,
        "kpi": kpi_name,

        "ref_mean":
        ref_values.mean(),

        "sim_mean":
        sim_values.mean(),

        "ref_median":
        ref_values.median(),

        "sim_median":
        sim_values.median(),

        "ref_std":
        ref_values.std(),

        "sim_std":
        sim_values.std(),

        "ref_p95":
        ref_values.quantile(0.95),

        "sim_p95":
        sim_values.quantile(0.95),

        "wasserstein":
        wasserstein_distance(
            ref_values,
            sim_values
        )
    }

In [16]:
# ==========================================================
# CASE LEVEL TURNAROUND VALIDATION
# ==========================================================

ref_cases = (
    ref.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

sim_cases = (
    sim.groupby(
        "case:concept:name"
    )["turnaround_time"]
    .max()
)

case_eval = pd.DataFrame(
    [
        {
            "kpi": "turnaround_time",

            "ref_mean":
            ref_cases.mean(),

            "sim_mean":
            sim_cases.mean(),

            "ref_median":
            ref_cases.median(),

            "sim_median":
            sim_cases.median(),

            "wasserstein":
            wasserstein_distance(
                ref_cases,
                sim_cases
            )
        }
    ]
)

case_eval

,kpi,ref_mean,sim_mean,ref_median,sim_median,wasserstein
0,turnaround_time,37.37715,55.379609,30.0,31.0,18.260615


In [17]:
# WHOLE EVALUATION
results = []
segments = {

    "all_rmg": (
        ref_rmg,
        sim_rmg
    ),

    "receive": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_receive"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_receive"
        ]
    ),

    "delivery": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_delivery"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_delivery"
        ]
    ),

    "mixed": (
        ref_rmg[
            ref_rmg["concept:name"]
            == "RMG_mixed"
        ],
        sim_rmg[
            sim_rmg["concept:name"]
            == "RMG_mixed"
        ]
    )
}

for segment, (
    ref_seg,
    sim_seg
) in segments.items():

    for kpi in [
        "waiting_time",
        "service_time",
        "turnaround_time"
    ]:

        results.append(

            evaluate_kpi(
                ref_seg[kpi],
                sim_seg[kpi],
                kpi,
                segment
            )

        )

evaluation = pd.DataFrame(
    results
)

In [18]:
# FINAL EVALUATION TABLE
evaluation = evaluation.round(2)

evaluation = evaluation.sort_values(
    [
        "segment",
        "kpi"
    ]
)

evaluation

,segment,kpi,ref_mean,sim_mean,ref_median,sim_median,ref_std,sim_std,ref_p95,sim_p95,wasserstein
1,all_rmg,service_time,11.71,16.25,8.0,9.0,11.43,98.09,32.00,33.00,4.63
2,all_rmg,turnaround_time,37.38,55.72,30.0,32.0,27.77,205.43,85.00,97.76,18.56
0,all_rmg,waiting_time,7.28,25.70,3.0,7.0,13.98,174.37,32.00,52.06,18.42
7,delivery,service_time,8.47,12.60,6.0,5.0,8.55,96.12,23.00,26.00,5.39
8,delivery,turnaround_time,29.69,57.97,23.0,29.0,25.27,236.43,67.45,92.26,28.62
6,delivery,waiting_time,6.08,32.39,3.0,7.0,9.03,216.40,24.00,56.15,26.31
10,mixed,service_time,14.86,23.52,11.0,11.0,13.38,140.48,39.00,46.00,9.12
11,mixed,turnaround_time,41.03,63.67,34.0,35.0,27.73,228.00,91.70,109.72,23.17
9,mixed,waiting_time,7.97,26.59,3.0,7.0,14.88,179.56,33.00,55.00,18.62
4,receive,service_time,12.15,15.04,9.0,9.0,11.53,85.85,32.00,31.00,3.53


In [19]:
# QUICK OVERVIEW ONLY WASSERSETEIN
evaluation.pivot(
    index="segment",
    columns="kpi",
    values="wasserstein"
).round(2)

kpi,service_time,turnaround_time,waiting_time
segment,,,
all_rmg,4.63,18.56,18.42
delivery,5.39,28.62,26.31
mixed,9.12,23.17,18.62
receive,3.53,16.12,17.10
